# AQUA-SHIELD — 2× Tesla T4 training notebook (Kaggle)

Runs the AQUA-SHIELD detector training + full evaluation pipeline on Kaggle's
free **2× Tesla T4** GPU accelerator. This notebook does not reimplement
anything — it drives the exact same scripts already built and tested on
Apple Silicon (`scripts/train.py`, `fit_verification.py`, `evaluate.py`,
`benchmark.py`, `robustness.py`, `export_edge.py`). Nothing here is a new
codepath; it's the same pipeline on faster, real CUDA hardware.

**Before running:**
1. Notebook settings → **Accelerator: GPU T4 x2**, **Internet: On**.
2. Upload the `aqua-shield/` project folder as a Kaggle **Dataset** (see
   Cell 2 for the exact zip command to run locally first), then **Add Data**
   → attach that dataset to this notebook. The repo is not currently a
   public GitHub repo, so `git clone` is not available as a fallback —
   dataset upload is the primary path.
3. (Optional, for the ghost-gear section only) Add your HuggingFace token as
   a **Kaggle Secret** named `HF_TOKEN` — never paste a token directly into a
   cell.

**Honesty note:** this notebook has been written and reviewed but **not
executed on Kaggle** — I do not have Kaggle access from this session. Every
command below reuses exactly the scripts already validated end-to-end on
Apple Silicon (116/116 tests passing, full pipeline runs verified). Treat the
first run as the first real test of the CUDA code path — `device.py`
already supports it (`cuda_available` fallback), but it has not been
exercised before now. Report back anything that breaks; nothing here should
need architecture changes, only environment fixes if any surface.


## 1 · Confirm the hardware

In [ ]:
!nvidia-smi


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  "
          f"({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")


## 2 · Get the code onto Kaggle

**The repo is now public** — `git clone` is the simplest path (used below).
A Kaggle Dataset upload is kept as a fallback in case Internet access is
restricted in your notebook session (some Kaggle competition/session modes
disable outbound network access even with GPU on).


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/aqua-shield")
REPO_URL = "https://github.com/Eartherai/SIH-2026.git"

if WORK.exists():
    shutil.rmtree(WORK)

cloned = False
try:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK)],
                   check=True, capture_output=True, text=True, timeout=120)
    cloned = True
    print("cloned via git:", REPO_URL)
except Exception as e:
    print(f"git clone failed ({type(e).__name__}: {e}) -- falling back to "
          "a Kaggle Dataset upload, if one is attached.")

if not cloned:
    candidates = list(Path("/kaggle/input").glob("*/aqua-shield")) + \
                 list(Path("/kaggle/input").glob("*/aqua_shield"))
    if not candidates:
        candidates = [p for p in Path("/kaggle/input").glob("*") if p.is_dir()
                      and (p / "src" / "aquashield").exists()]
    assert candidates, (
        "git clone failed AND no dataset found under /kaggle/input/. "
        "Either allow Internet access in notebook settings, or upload the "
        "aqua-shield/ folder as a Kaggle Dataset (zip command in the markdown "
        "above this cell) and attach it via 'Add Data'."
    )
    shutil.copytree(candidates[0], WORK)
    print("copied from Kaggle Dataset at:", candidates[0])

os.chdir(WORK)
print("cwd:", os.getcwd())


## 3 · Install dependencies

Kaggle's base image already ships a CUDA-matched `torch`/`torchvision` and
`numpy`/`scipy`/`pandas`/`opencv`/`scikit-learn` — we deliberately do **not**
reinstall those (forcing a different torch build risks breaking the CUDA
linkage Kaggle already configured correctly). Everything else in
`requirements.txt` is installed as-is, including the same
`ultralytics<8.4` pin that was found necessary on Apple Silicon (8.4.x
diverged during training on this exact dataset — see
`docs/BENCHMARKS.md` §8 — the pin is dataset-specific, not
hardware-specific, so it is kept here too).


In [ ]:
!pip install -q "ultralytics>=8.3,<8.4" pyproj tifffile shapely \
    streamlit folium streamlit-folium plotly \
    fastapi uvicorn python-multipart pydantic \
    onnx onnxslim "onnxruntime-gpu>=1.19"


In [ ]:
import sys
sys.path.insert(0, "src")
from aquashield.device import environment_report
print(environment_report())


## 4 · Download and prepare MILCO/NOMBO

Identical to the local pipeline: CC BY 4.0, ~218 MB, survey-level
(leakage-free) splits by acquisition year. Requires **Internet: On**.


In [ ]:
!python scripts/download_datasets.py


In [ ]:
!python scripts/prepare_milco_nombo.py


## 5 · Train — same recipe as the primary model (E04), on real CUDA

This reproduces the exact recipe behind `aquashield_primary.pt`
(`docs/BENCHMARKS.md` §2), but with headroom Apple Silicon didn't have:
larger batch size (T4 has 16 GB vs. the M5's shared unified memory) and no
epoch-budget compromise from wall-clock pressure. `--device auto` resolves
to `cuda` automatically (`aquashield.device.select_device` — MPS
unavailable, CUDA detected).


In [ ]:
!python scripts/train.py --exp-id E10-kaggle-t4-primary-recipe \
  --device auto \
  --epochs 150 --imgsz 640 --batch 32 --lr0 0.002 --cos-lr --patience 40 \
  --mosaic 0.3 --scale 0.25 --fliplr 0.5 --flipud 0.5 \
  --hsv-h 0.0 --hsv-s 0.0 --hsv-v 0.4 \
  --notes "Kaggle 2xT4: E04's exact recipe re-run on real CUDA with batch 32 (vs 16 on M5) and no wall-clock-driven early stop. Directly comparable to E04 for a same-recipe cross-hardware check."


## 6 · Optional — test whether more detector capacity helps

Every version of this project has *reasoned* that YOLO11s/m weren't worth
attempting on 447 training objects, but reasoning is not the same as
measuring, and that reasoning was partly a wall-clock/compute-budget call
made on the M5. On a real GPU this experiment is cheap enough to just run
and report honestly — whether it helps, hurts, or overfits.


In [ ]:
# Optional: swap the base model. scripts/train.py accepts any Ultralytics
# checkpoint name via --model.
!python scripts/train.py --exp-id E11-kaggle-yolo11s \
  --model yolo11s.pt \
  --device auto \
  --epochs 150 --imgsz 640 --batch 24 --lr0 0.002 --cos-lr --patience 40 \
  --mosaic 0.3 --scale 0.25 --fliplr 0.5 --flipud 0.5 \
  --hsv-h 0.0 --hsv-s 0.0 --hsv-v 0.4 \
  --notes "First-ever test of YOLO11s (vs primary's YOLO11n) on this dataset. Previously reasoned-not-attempted due to M5 compute budget; now measured on a real GPU instead of assumed. Compare mAP50/recall against E10 honestly -- more capacity on 447 objects may overfit rather than help."


## 7 · Optional — actually use BOTH T4s in one run

`scripts/train.py` calls `aquashield.device.select_device()`, which resolves
to a single device string (`"cuda"`, i.e. GPU 0) — it does not know about
multi-GPU indices, because that logic doesn't exist on Apple Silicon and was
never built. To genuinely exercise both T4s in one DataParallel run, call
Ultralytics directly with `device=[0,1]`, bypassing the wrapper. This is the
one cell in this notebook that does **not** go through our own scripts —
flagged here explicitly so that distinction isn't lost.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(
    data="data/processed/milco_nombo_yolo/data.yaml",
    epochs=150, imgsz=640, batch=64,           # batch scaled up for 2 GPUs
    device=[0, 1],                              # BOTH T4s, DataParallel
    lr0=0.002, cos_lr=True, patience=40, seed=0, deterministic=True,
    degrees=0.0, fliplr=0.5, flipud=0.5, mosaic=0.3, scale=0.25,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.4,
    project="runs/train", name="E12-kaggle-dual-t4", exist_ok=True,
)
# NOTE: this run is NOT auto-recorded in experiments/registry.jsonl the way
# scripts/train.py runs are (that logic lives in the wrapper this cell
# bypasses). If you keep this run, log its metrics into the registry
# manually using the same schema as the other entries -- see Cell 10.


## 8 · Full evaluation suite

Reuses `scripts/run_full_evaluation.sh` exactly — pipeline ablation,
latency/memory benchmark (now against a real GPU instead of MPS/CPU),
robustness under controlled perturbation, and ONNX export. **Step 1 of this script is the verification fit** (false-positive filter + calibration) under the standard `milco_nombo` tag every other part of the pipeline expects -- no separate fit step is needed. Regenerates the
same `docs/BENCHMARKS.md`-style tables from fresh `experiments/*.json`.


In [ ]:
!bash scripts/run_full_evaluation.sh \
  runs/detect/runs/train/E10-kaggle-t4-primary-recipe/weights/best.pt


In [ ]:
# GPU-specific latency: benchmark.py already tries every available device
# (mps, cpu) -- on Kaggle it will try mps (absent, skipped) and cpu. Run cuda
# explicitly for the number that actually matters here.
!python scripts/benchmark.py \
  --weights runs/detect/runs/train/E10-kaggle-t4-primary-recipe/weights/best.pt \
  --devices cuda,cpu --n 30


## 9 · Compare against the Apple Silicon numbers, honestly

Read straight from `experiments/registry.jsonl` — don't eyeball printed
epoch logs. Floating-point paths differ between MPS and CUDA (documented
caveat in `docs/BENCHMARKS.md` §9: "MPS kernels are not bit-reproducible
across PyTorch versions" — the same applies across backends), so expect
small differences even given an identical recipe; a large difference is a
real finding worth reporting, not an error to explain away.


In [ ]:
import json
for line in open("experiments/registry.jsonl"):
    r = json.loads(line)
    if r["experiment_id"] in ("E04-smallobj-tuned", "E10-kaggle-t4-primary-recipe",
                              "E11-kaggle-yolo11s"):
        m = r["metrics_test"]
        print(f"{r['experiment_id']:32s} device={r['hardware']['device']:5s} "
              f"mAP50={m['mAP50']:.4f} P={m['precision']:.4f} R={m['recall']:.4f}")


## 10 · Package results for download

Kaggle sessions are ephemeral (9h max, weekly GPU quota) — nothing here
persists after the session ends unless you download it. This zips the
checkpoints, the experiment registry, and the fitted verification files so
you can merge them back into the local repo's `models/` and `experiments/`
the same way every other experiment in this project was recorded.


In [ ]:
import shutil
from pathlib import Path

out = Path("/kaggle/working/aqua_shield_kaggle_results")
out.mkdir(exist_ok=True)

for pattern in ["runs/detect/runs/train/*/weights/best.pt",
               "runs/detect/runs/val/*/results.csv",
               "models/fp_filter_*.json", "models/calibration_*.json",
               "models/verification_fit_*.json",
               "experiments/registry.jsonl", "experiments/ablation.json",
               "experiments/benchmarks.jsonl", "experiments/robustness.json",
               "experiments/edge_export.json"]:
    for f in Path(".").glob(pattern):
        dest = out / f.relative_to(".")
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dest)

shutil.make_archive("/kaggle/working/aqua_shield_kaggle_results", "zip", out)
print("wrote /kaggle/working/aqua_shield_kaggle_results.zip")
print("Download it from the Kaggle notebook's Output panel.")


## 11 · Optional — ghost-gear (crab-pot) training

Only runs if you added `HF_TOKEN` as a **Kaggle Secret** (Add-ons → Secrets)
*and* you have already clicked "Agree and access repository" on
`https://huggingface.co/datasets/PINGEcosystem/sss-crab-pot-detection-ds` —
the dataset is gate-`auto` (instant on that click) but the click itself is a
human action nothing here can perform. If access has not been granted yet,
`prepare_crab_pot.py` will fail fast with exactly that explanation rather
than retry or hang.


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as e:
    print("No HF_TOKEN secret found or accessible -- skip this section, "
          "or add it under Add-ons > Secrets and re-run this cell.")
    print(f"  ({type(e).__name__}: {e})")


In [ ]:
!python scripts/prepare_crab_pot.py


In [ ]:
!python scripts/train.py --exp-id E13-kaggle-crab-pot \
  --data data/processed/crab_pot_yolo/data.yaml \
  --device auto \
  --epochs 150 --imgsz 640 --batch 32 --lr0 0.002 --cos-lr --patience 40 \
  --mosaic 0.3 --scale 0.25 --fliplr 0.5 --flipud 0.5 \
  --hsv-h 0.0 --hsv-s 0.0 --hsv-v 0.4 \
  --notes "First-ever ghost-gear (derelict crab pot) training run, contingent on manual HF dataset access. Closes the single biggest documented gap in the project (README.md SS4.3): AQUA-SHIELD had never been trained on the problem statement's actual named target class."
